## Probabilistic Spelling Corrector

Adapted from Peter Norvig's design: [How to Write a Spelling Corrector](http://norvig.com/spell-correct.html)

**Original work:**
- Copyright (c) 2007–2016 Peter Norvig
- Licensed under the [MIT License](https://opensource.org/licenses/MIT)

**Modifications:** Diego Besada

\* Please note that this implementation is not intended to be efficient but legible and easy to understand.

In [1]:
import re, requests, time
from collections import Counter

#### 1. Corpus Tokenization

We will first define a function that takes a text string and returns a Counter object of the words in that text. We use a regular expression instead of the built-in `split()` method to properly handle punctuation and whitespace. The function also converts all words to lowercase to ensure the spell checker is case-insensitive. This can be seen as a form of tokenization, where we break down the text into individual tokens (words) for further processing.

In [2]:
def get_words(text: str) -> Counter[str]:
    '''Count the frequency of each word in the text.'''
    return Counter(re.findall(r'\w+', text.lower()))

#### 2. Candidate Generation ([Damerau–Levenshtein Distance](https://en.wikipedia.org/wiki/Damerau–Levenshtein_distance))

For creating candidate words, we will use the concept of edit distance. The edit distance between two words is defined as the minimum number of operations required to transform one word into the other. The operations we will consider are insertion, deletion, replacement, and transposition of adjacent characters.

Once we have an edit of a word, we can also make edits of that edit, and so on. This allows us to generate candidates that are multiple edits away from the original word. I have used recursion to generate candidates that are up to `n` edits away from the original word.

In [3]:
def get_edits(word: str, n: int = 1) -> set[str]:
    '''Creates a set of all possible edits that are one edit away from `word`'''

    def _get_edits1(word: str) -> set[str]:
        results = set()
        letters = 'abcdefghijklmnopqrstuvwxyz'

        for i in range(len(word) + 1):
            left, right = word[:i], word[i:]

            # Insertions
            for l in letters:
                results.add(left + l + right)

            if right:
                # Deletions
                results.add(left + right[1:])
                
                # Replacements
                for c in letters:
                    results.add(left + c + right[1:])

            # Transpositions
            if len(right) > 1:
                results.add(left + right[1] + right[0] + right[2:])

        return results

    if n == 0:
        return {word}
    return {e2 for e1 in get_edits(word, n - 1) for e2 in _get_edits1(e1)}

#### 3. Vocabulary Validation

Once we have generated a set of candidate words, we need to check which of these candidates are actually valid words in our dictionary.

In [4]:
def is_known(words: set[str] | str, corpus: set[str]) -> set[str]:
    '''Returns the subset of `words` that appear in the `corpus`.'''
    if isinstance(words, str):
        words = {words}
    return words & corpus

#### 4. Candidate Retrieval Hierarchy

Candidates are selected using a tiered priority order based on minimum edit distance.

In [5]:
def get_candidates(word: str, corpus: set[str]) -> set[str]:
    '''Generate possible spelling corrections for word.'''
    return (is_known(word, corpus) or is_known(get_edits(word), corpus) or is_known(get_edits(word, 2), corpus) or {word})

#### 5. Maximum A Posteriori (MAP) Selection

Select the candidate $c$ that maximizes the prior probability $P(c)$, approximated by its raw occurrence count in the corpus.

In [6]:
def make_correction(word: str, corpus: Counter[str]) -> str: 
    '''Most probable spelling correction for word.'''
    return max(get_candidates(word, set(corpus)), key=corpus.get)

### **TESTING**

We have provided all the necessary functions to implement a spelling corrector. Now we will test the corrector with some example words.

In [7]:
# Load the corpus and create a Counter of words
CORPUS = requests.get('https://norvig.com/big.txt').text
WORDS = Counter(get_words(CORPUS))

# Make a correction for different words given the corpus
make_correction('speling', WORDS)        # 'spelling'       | insert
make_correction('korrectud', WORDS)      # 'corrected'      | replace 2
make_correction('bycycle', WORDS)        # 'bicycle'        | replace
make_correction('inconvient', WORDS)     # 'inconvenient'   | insert 2
make_correction('arrainged', WORDS)      # 'arranged'       | delete
make_correction('peotry', WORDS)         # 'poetry'         | transpose
make_correction('peotryy', WORDS)        # 'poetry'         | transpose + delete
make_correction('word', WORDS)           # 'word'           | known
make_correction('quintessential', WORDS) # 'quintessential' | unknown

'quintessential'

In the original implementation, the corrector relied on a global variable for the vocabulary. I chose to pass the corpus as an explicit argument to make the function pure and flexible, allowing it to work with different corpora without code changes.

It is also worth noting that this is a simple probabilistic model based solely on word frequencies. Because it does not account for grammar or sentence context, its suggestions can be limited and are heavily influenced by the specific domain of the source corpus.

Now we are going to test the corrector with a set of test cases. The test cases are provided in a list of tuples, where each tuple contains a correct word and a misspelled version of that word. The `spelltest` function will run the corrector on each misspelled word and report the results.

In [8]:
def spelltest(tests: list[tuple[str, str]], corpus: Counter[str]) -> None:
    'Run correction(wrong) on all (right, wrong) pairs; report results.'
    start = time.perf_counter()
    good, unknown = 0, 0
    n = len(tests)

    for right, wrong in tests:
        w = make_correction(wrong, corpus)
        good += (w == right)
        if w != right:
            unknown += (right not in corpus)
            
    dt = time.perf_counter() - start
    print('{:.0%} of {} correct ({:.0%} unknown) at {:.0f} words per second '
          .format(good / n, n, unknown / n, n / dt))
    
def Testset(lines: list[str]) -> list[tuple[str, str]]:
    'Parse "right: wrong1 wrong2" lines into [("right", "wrong1"), ("right", "wrong2")] pairs.'
    return [(right, wrong)
            for (right, wrongs) in (line.split(':') for line in lines)
            for wrong in wrongs.split()]

In [9]:
testset1 = requests.get('https://norvig.com/spell-testset1.txt').text
testset2 = requests.get('https://norvig.com/spell-testset2.txt').text

spelltest(Testset(testset1.splitlines()), WORDS)

75% of 270 correct (6% unknown) at 88 words per second 


In [10]:
spelltest(Testset(testset2.splitlines()), WORDS) 

68% of 400 correct (11% unknown) at 78 words per second 
